# Prioritization of Ligands Based on Expression Values

In this notebook, we extend the basic NicheNet analysis by incorporating gene expression as part of the prioritization. While the original NicheNet only ranks ligands based on the ligand activity analysis, it is now also possible to prioritize ligands based on cell type and condition specificity of the ligand and receptor.

We again use mouse NICHE-seq data to explore intercellular communication in the T cell area in the inguinal lymph node before and 72 hours after LCMV infection (Medaglia et al., 2017).

In [ ]:
import os
os.environ.setdefault("NICHENETR_DATA_DIR", "path/to/nichenetr_data")

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import nichenetr as nn

## Prepare NicheNet Analysis

Load data and networks.

In [ ]:
adata = nn.load_seurat_obj()
adata = nn.alias_to_symbol_anndata(adata, "mouse")

lr_network = nn.load_lr_network("mouse")
ligand_target_matrix = nn.load_ligand_target_matrix("mouse")
weighted_networks = nn.load_weighted_networks("mouse")

lr_network = lr_network[["from", "to"]].drop_duplicates()

## Perform the NicheNet Analysis (sender-focused)

In [ ]:
# 1. Define set of potential ligands
receiver = "CD8 T"
expressed_genes_receiver = nn.get_expressed_genes(
    adata, celltype_col="celltype", celltype=receiver, pct=0.05
)

sender_celltypes = ["CD4 T", "Treg", "Mono", "NK", "B", "DC"]
expressed_genes_sender = set()
for ct in sender_celltypes:
    expressed_genes_sender.update(
        nn.get_expressed_genes(adata, celltype_col="celltype", celltype=ct, pct=0.05)
    )
expressed_genes_sender = list(expressed_genes_sender)

all_ligands = lr_network["from"].unique().tolist()
all_receptors = lr_network["to"].unique().tolist()

expressed_ligands = list(set(all_ligands) & set(expressed_genes_sender))
expressed_receptors = list(set(all_receptors) & set(expressed_genes_receiver))

potential_ligands = (
    lr_network[
        lr_network["from"].isin(expressed_ligands) & lr_network["to"].isin(expressed_receptors)
    ]["from"].unique().tolist()
)

In [ ]:
# 2. Define gene set of interest
condition_oi = "LCMV"
condition_reference = "SS"

receiver_mask = adata.obs["celltype"] == receiver
adata_receiver = adata[receiver_mask].copy()

sc.tl.rank_genes_groups(
    adata_receiver, groupby="aggregate",
    groups=[condition_oi], reference=condition_reference, method="wilcoxon",
)
de_table = sc.get.rank_genes_groups_df(adata_receiver, group=condition_oi)
de_table = de_table.rename(columns={"names": "gene", "logfoldchanges": "avg_log2FC", "pvals_adj": "p_val_adj"})

geneset_oi = de_table[
    (de_table["p_val_adj"] <= 0.05) & (de_table["avg_log2FC"].abs() >= 0.25)
]["gene"].tolist()
geneset_oi = [g for g in geneset_oi if g in ligand_target_matrix.rownames]

# 3. Background genes
background_expressed_genes = [g for g in expressed_genes_receiver if g in ligand_target_matrix.rownames]

In [ ]:
# 4. Ligand activity analysis
ligand_activities = nn.predict_ligand_activities(
    geneset=geneset_oi,
    background_expressed_genes=background_expressed_genes,
    ligand_target_matrix=ligand_target_matrix,
    potential_ligands=potential_ligands,
)

ligand_activities = ligand_activities.sort_values("aupr_corrected", ascending=False)
ligand_activities["rank"] = ligand_activities["aupr_corrected"].rank(ascending=False)
ligand_activities.head(10)

## Perform Prioritization of Ligand-Receptor Pairs

We prioritize ligand-receptor pairs based on:
- Upregulation of the ligand in a sender cell type (`de_ligand`)
- Upregulation of the receptor in a receiver cell type (`de_receptor`)
- Average expression of ligand/receptor (`exprs_ligand`, `exprs_receptor`)
- Condition-specificity of ligand/receptor (`ligand_condition_specificity`, `receptor_condition_specificity`)

### Using the wrapper function `generate_info_tables`

In [ ]:
lr_network_filtered = lr_network[
    lr_network["from"].isin(expressed_ligands) & lr_network["to"].isin(expressed_receptors)
]

info_tables = nn.generate_info_tables(
    adata,
    celltype_colname="celltype",
    senders_oi=sender_celltypes,
    receivers_oi=receiver,
    lr_network=lr_network_filtered,
    condition_colname="aggregate",
    condition_oi=condition_oi,
    condition_reference=condition_reference,
    scenario="case_control",
)

print("Info table keys:", list(info_tables.keys()))
print("\nsender_receiver_de:")
print(info_tables["sender_receiver_de"].head())
print("\nsender_receiver_info:")
print(info_tables["sender_receiver_info"].head())
print("\nlr_condition_de:")
print(info_tables["lr_condition_de"].head())

In [ ]:
# Generate prioritization table
prior_table = nn.generate_prioritization_tables(
    info_tables["sender_receiver_info"],
    info_tables["sender_receiver_de"],
    ligand_activities,
    info_tables["lr_condition_de"],
    scenario="case_control",
)

prior_table.head()

The resulting table shows the rankings for ligand-receptor interactions of a sender-receiver cell type pair.

In [ ]:
# Show relevant ranking columns
cols_oi = [
    "sender", "receiver", "ligand", "receptor",
    "scaled_p_val_adapted_ligand", "scaled_p_val_adapted_receptor",
    "scaled_avg_exprs_ligand", "scaled_avg_exprs_receptor",
    "scaled_activity",
]
cols_available = [c for c in cols_oi if c in prior_table.columns]
prior_table[cols_available].head(10)

### Step-by-step prioritization

You can also compute the components separately for more flexibility.

In [ ]:
# Calculate DE for the condition of interest
DE_table = nn.calculate_de(
    adata,
    celltype_col="celltype",
    condition_oi=condition_oi,
    condition_col="aggregate",
)

# Average expression
expression_info = nn.get_exprs_avg(
    adata, celltype_col="celltype",
    condition_colname="aggregate", condition_oi=condition_oi,
)

# Process tables for prioritization
processed_DE = nn.process_table_to_ic(
    DE_table, table_type="celltype_DE", lr_network=lr_network_filtered,
    senders_oi=sender_celltypes, receivers_oi=receiver,
)

processed_expr = nn.process_table_to_ic(
    expression_info, table_type="expression", lr_network=lr_network_filtered,
)

In [ ]:
# Custom weights
prioritizing_weights = {
    "de_ligand": 1,
    "de_receptor": 1,
    "activity_scaled": 1,
    "exprs_ligand": 1,
    "exprs_receptor": 1,
    "ligand_condition_specificity": 1,
    "receptor_condition_specificity": 1,
}

## Prioritizing across multiple receivers

To prioritize ligand-receptor pairs across multiple receivers, perform the analysis for each receiver separately then combine.

In [ ]:
nichenet_outputs = {}
for receiver_ct in ["CD8 T", "CD4 T", "Treg"]:
    output = nn.nichenet_seuratobj_aggregate(
        receiver=receiver_ct,
        adata=adata,
        condition_col="aggregate",
        condition_oi=condition_oi,
        condition_ref=condition_reference,
        sender=sender_celltypes,
        celltype_col="celltype",
        ligand_target_matrix=ligand_target_matrix,
        lr_network=lr_network,
        weighted_networks=weighted_networks,
        expression_pct=0.05,
    )
    output["ligand_activities"]["receiver"] = receiver_ct
    nichenet_outputs[receiver_ct] = output

In [ ]:
# Combine ligand activities and generate combined prioritization
ligand_activities_combined = pd.concat(
    [o["ligand_activities"] for o in nichenet_outputs.values()],
    ignore_index=True,
)
ligand_activities_combined.head()

### Visualization: Circos LR plot

In [ ]:
# Get top 50 ligand-receptor pairs
if "prioritization_score" in prior_table.columns:
    prior_table_oi = prior_table.nlargest(50, "prioritization_score")
    
    # Define colors for senders and receivers
    senders_receivers = sorted(
        set(prior_table_oi["sender"].tolist() + prior_table_oi["receiver"].tolist())
    )
    import matplotlib.cm as cm
    colors = cm.Set3(np.linspace(0, 1, len(senders_receivers)))
    celltype_colors = {ct: colors[i] for i, ct in enumerate(senders_receivers)}
    
    nn.make_circos_lr(
        prior_table_oi,
        colors_sender=celltype_colors,
        colors_receiver=celltype_colors,
        show=True,
    )

### Visualization: Mushroom plot

In [ ]:
receiver_oi = "CD8 T"
if "receiver" in prior_table.columns:
    nn.make_mushroom_plot(
        prior_table[prior_table["receiver"] == receiver_oi],
        top_n=30,
        show=True,
    )